# ASH-KV + SGLang: T4 Validation
This notebook is designed to test the ASH-KV INT8 Shadow Cache integration with SGLang on a T4 GPU (Colab/Kaggle). Since notebook cells run sequentially, we start the server as a background process.

In [ ]:
!git clone https://github.com/Ijas14/ASH-KV.git
%cd ASH-KV

import os
import urllib.request
import json

# 1. Fetch SGLang dependencies and filter out the CUDA/PyTorch ones that break Colab
print("Resolving safe SGLang dependencies...")
reqs = json.loads(urllib.request.urlopen("https://pypi.org/pypi/sglang/json").read())["info"]["requires_dist"]
safe_reqs = []
for req in reqs:
    if "extra ==" in req: continue  # skip extras like [all]
    clean_name = req.split(';')[0].strip().lower()
    if any(bad in clean_name for bad in ['torch', 'triton', 'cuda', 'nvidia', 'flash', 'sgl-deep-gemm', 'vllm']):
        continue
    # Extract just the package name without version pins for maximum compatibility
    pkg_name = req.split(';')[0].split('>=')[0].split('==')[0].split('<')[0].strip()
    if pkg_name == 'transformers': pkg_name = 'transformers==4.44.2'
    if pkg_name: safe_reqs.append(pkg_name)

print(f"Installing {len(safe_reqs)} safe dependencies...")
os.system(f"pip install {' '.join(safe_reqs)}")

# 2. Install SGLang core WITHOUT dependencies
!pip install "sglang>=0.3.0" sglang-router --no-deps

# 3. Install ASH-KV WITHOUT dependencies
!pip install -e . --no-deps

# 4. Dynamically fetch the correct FlashInfer wheel


In [ ]:
import torch
import triton
import sglang
try:
    import flashinfer
    has_fi = True
except:
    has_fi = False

print(f"PyTorch Version: {torch.__version__}")
print(f"Triton Version: {triton.__version__}")
print(f"SGLang Version: {sglang.__version__}")
if has_fi:
    print(f"FlashInfer Version: {flashinfer.__version__}")
else:
    print("FlashInfer: Not Installed (Using Triton Backend)")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import time
os.environ['ASHKV_ENABLED'] = '1'
!pip uninstall -y kernels
!pip install transformers==4.44.2

print('Starting SGLang server in the background...')
!nohup python3 -m sglang.launch_server --model-path Qwen/Qwen3.5-2B --mem-fraction-static 0.4 --port 30000 --attention-backend triton > server.log 2>&1 &

print('Waiting for weights to load (60 seconds)...')
time.sleep(60)
print('Server booted! Check server.log for details.')


In [ ]:
# Verify the server started successfully without crashing
!tail -n 50 server.log

In [ ]:
# STRESS TEST: Fire 32 concurrent requests at the server.
# Because we restricted memory to 0.4, this will trigger the SGLang eviction logic.
# The ASH-KV hooks will intercept these evictions, compress to INT8, and prevent OOM.
!python3 -m sglang.bench_serving --backend sglang --num-prompts 32 --port 30000

In [ ]:
# Check the logs for telemetry and circuit breaker stats after the run
!tail -n 100 server.log

In [ ]:
# Cleanup
server_process.terminate()
print("Server stopped.")